# Which discharge practices matter: ranking candidate exposures with TMLE

## At a glance

Hospital teams chose which of five discharge practices each patient going home got. Which of them
change the chance that a patient stays home, with no readmission, for 30 days? The page estimates
one effect for each practice with targeted maximum likelihood estimation (TMLE), which combines a
model of the outcome with a model of who got the practice. It then adjusts the p-values for testing
five practices at once. The data are simulated, so each estimate can be compared with a known answer.

Booking a follow-up visit before discharge would raise the share who stay home by 0.089, about 9
percentage points (95% interval 0.062 to 0.117). A pharmacist's medication review would raise it
by 0.044 (0.017 to 0.071), and offering transition navigation by 0.059 (0.032 to 0.086). Neither
teach-back education nor a home-health referral shows an effect, and in these data neither has one.

Each answer holds only if nothing unrecorded drove both that practice and staying home, and in
this sample navigation's answer would not survive a hidden factor as strong as discharge risk.
Navigators book the follow-up visits, and the 0.059 counts the visits the offer brings. The tool's
default fit holds each patient's booking as it was, so it leaves those visits out and makes
navigation look weak (0.030, adjusted p-value 0.0794).

## The question

The health plan wants to know which practices to make standard for every patient going home. For
each practice the quantity is the change in the share of patients who stay home 30 days if every
patient got that practice, compared with no patient getting it: an average treatment effect, one
for each candidate practice (a *candidate exposure*).

```text
  (an arrow means "causes")
  W ──▶ every practice, and W ──▶ Y

  navigation ──▶ follow-up booked ──▶ Y
  navigation ──────────────────────▶ Y
  medication review ───────────────▶ Y
  teach-back          (no arrow to Y)
  home-health referral (no arrow to Y)

  W  = discharge_risk, age, prior_utilization
  Y  = stayed_home (no readmission by day 30)
  assumed absent: an unrecorded cause U with U ──▶ a practice and U ──▶ Y
```

The three recorded characteristics in W change both who gets each practice and the outcome, so
every fit adjusts for them. Navigators book follow-up visits, so the offer changes the booking,
and part of its effect runs through the visit. The missing U, an unrecorded *confounder*, is the key
assumption, and it is one assumption per practice: nothing unrecorded drove both that practice and
staying home.

## The data

Every fit sets its seeds, so a rerun with the version printed below gives the same numbers.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

`make_discharge_practices` draws 4,000 simulated discharges and the true effect of each practice,
which real data never give.

In [ ]:
from cleverly.datasets import PRACTICES, make_discharge_practices

# With your own data, load your data frame here, such as frame = pd.read_csv(...).
# Real data have no `truth`, so drop every line that uses it.
frame, truth = make_discharge_practices(n=4_000, seed=0)
# For your own data, change these lists: the practices to rank, and the characteristics
# recorded before any of them.
candidates = list(PRACTICES)
covariates = ["discharge_risk", "age", "prior_utilization"]
print("rows and columns:", frame.shape)
print(list(frame.columns))
print()
true_effects = pd.DataFrame(
    {
        "share who get it": [truth[f"share[{name}]"] for name in candidates],
        "others held as they were": [truth[f"adjusted[{name}]"] for name in candidates],
        "booking follows the offer": [truth[f"total[{name}]"] for name in candidates],
    },
    index=candidates,
)
print("true effects on the share who stay home:")
print(true_effects.round(4))

Each row is one discharge: whether the patient stayed home (1 or 0), the five practices (1 where
the patient got it), and three characteristics measured before discharge, on a standardized scale.
Two columns of true effects differ only for navigation: 0.0165 with each patient's booking held as
it was, and 0.0506 when the booking follows the offer. Teach-back and the home-health referral
have no effect.

## Why a simple comparison misleads

Teams chose who got each practice, so patients who got one can differ before discharge from those
who did not.

In [ ]:
simple = pd.DataFrame(
    {
        "simple difference": [
            frame.loc[frame[name] == 1, "stayed_home"].mean()
            - frame.loc[frame[name] == 0, "stayed_home"].mean()
            for name in candidates
        ],
        "true effect": [truth[f"total[{name}]"] for name in candidates],
        "mean discharge_risk, got it": [
            frame.loc[frame[name] == 1, "discharge_risk"].mean() for name in candidates
        ],
        "mean discharge_risk, did not": [
            frame.loc[frame[name] == 0, "discharge_risk"].mean() for name in candidates
        ],
    },
    index=candidates,
)
print(simple.round(3))

Patients referred to home health stayed home less often (a difference of -0.133), though the
referral does nothing, and navigation looks harmful (-0.028) though it helps. Teams referred and
offered navigation more often to patients with a higher `discharge_risk` (0.580 against -0.282 for
the referral), and those patients are readmitted more anyway. A fair comparison adjusts for the
characteristics.

## Set up the analysis

First, describe the study (`StudyProtocol`). The description changes no estimate: the ranking
records it, and `summary(detail="full")` prints it. One description covers all five practices,
each compared in turn with usual discharge without it.

In [ ]:
from cleverly.datasets import discharge_practices_protocol

# For your own study, fill in the same ten fields with cleverly.StudyProtocol(...),
# in the order the output prints them.
protocol = discharge_practices_protocol()
print("\n".join(protocol.summary_lines()))

`variable_importance` takes the outcome, the candidate practices and the characteristics to adjust
for, and fits each practice in turn as the treatment. By default each fit also adjusts for the
other four practices. That default is right when no practice causes another: a practice often
given together with another would otherwise take credit for the other's effect. Navigation causes
bookings, so this page runs a second version below. `variable_importance_plan` counts the model
fits before fitting anything. Each row is a causal effect, not a predictive feature importance
like a random forest's.

In [ ]:
from cleverly import variable_importance_plan
from cleverly.estimators import TMLE

# Left out, the learners default to a Super Learner, which costs 30 to 120 seconds per fit.
# This page passes logistic regressions. Each model is fitted on four of five folds of patients
# and predicts the fifth; five folds is a common choice, and flexible learners need them.
estimator = TMLE(
    outcome_learner=LogisticRegression(max_iter=1000),
    treatment_learner=LogisticRegression(max_iter=1000),
    n_folds=5,
    random_state=0,
)
arguments = {
    "outcome": "stayed_home",  # For your own data, change the outcome name.
    "candidates": candidates,
    "covariates": covariates,
    "estimator": estimator,
    "protocol": protocol,
}
planned = variable_importance_plan(frame, **arguments)
print("model fits this will run:", planned.total_fits)

Five fits of ten models each (an outcome and a treatment model in each of five folds) make 50; with
slow learners or many candidates, the count tells you the cost before you run it. Each practice's
answer is causal only if its own conditions hold:

- no unrecorded factor drove both that practice and staying home (this needs judgment);
- every kind of patient had some chance of getting the practice and of not getting it, called
  *overlap* or *positivity* (the data can check this, in part);
- each practice means the same thing for every patient, and one patient's practices do not change
  another's outcome (these need judgment).

Navigation's fit adjusts for the follow-up booking, which the offer causes, so it needs one more:
no unrecorded factor drove both whether a visit was booked and staying home (this needs judgment).

## Rank the practices

Each fit uses targeted maximum likelihood estimation (TMLE). It fits two models: an outcome model,
which predicts staying home from the practices and the characteristics, and a treatment model,
which predicts each patient's chance of getting the practice. A correction step then uses the
treatment model to remove the bias the outcome model leaves, weighting each patient by one over
their chance of the option they got. Each patient's predictions come from models trained on the
other four folds of patients, so a flexible model cannot grade its own training rows, which would
make the interval too narrow ([flexible models](cross-fitting.ipynb) shows this).

In [ ]:
from cleverly import variable_importance

ranking = variable_importance(frame, **arguments)
# The summary gives each practice's estimate, standard error, 95% interval, p-value for a zero
# effect, and that p-value adjusted for testing five practices.
print(ranking.summary())
print()
table = ranking.to_frame().set_index("candidate")
table["true effect"] = [truth[f"adjusted[{name}]"] for name in table.index]
table["covers truth"] = table["ci_lower"].le(table["true effect"]) & table["ci_upper"].ge(
    table["true effect"]
)
plain_names = {"psi": "estimate", "ci_lower": "95% low", "ci_upper": "95% high"}
shown = ["psi", "ci_lower", "ci_upper", "true effect", "covers truth"]
print(table[shown].rename(columns=plain_names).round(4))
print()
print("navigation's fit adjusts for:", table.loc["transition_navigation", "adjustment_set"])

Booking a follow-up visit comes first, then the medication review; teach-back and the home-health
referral sit near zero, with adjusted p-values of 0.7214. Navigation's raw p-value is 0.0477, but
adjusted it is 0.0794, so this run would not call it an effect. The true values here are those of
the "others held as they were" column, and in this sample each of the five intervals contains its
own.

The rows are sorted by adjusted p-value, so the first row is the clearest effect and need not be
the largest: compare the estimates and intervals to judge size. The adjustment keeps the *false
discovery rate*, the expected share of practices with no effect among those called significant, at
or below 5% when the five tests are independent or move together. These fits share their patients
and nothing guarantees that (note [2]), so treat the adjusted p-values as a guide to which leads to
follow up. Note [3] calls overlap "positivity".

## When one practice leads to another

Navigation's row above answers "what if we changed the offer while each patient's booking stayed as
it was?". The plan's question is what offering navigation would do, bookings included. The second
run stops adjusting each fit for the other practices (`adjust_for_other_candidates=False`) and adds
the offer to the characteristics. The offer changes both the booking and staying home, so it is a
confounder for the booking, and the booking's fit must still adjust for it.
A practice is never adjusted for in its own fit, so navigation's fit adjusts for the three
characteristics only.

In [ ]:
offer_first = variable_importance(
    frame,
    # For your own data, add here any practice that comes before others and can cause them.
    **{**arguments, "covariates": [*covariates, "transition_navigation"]},
    adjust_for_other_candidates=False,
)
second = offer_first.to_frame().set_index("candidate")
rows = []
for label, fitted, key in (
    ("others held", table, "adjusted"),
    ("booking follows", second, "total"),
):
    # For your own data, name the practices whose two runs you want to compare.
    for name in ("transition_navigation", "followup_booked"):
        row = fitted.loc[name]
        rows.append(
            {
                "fit": label,
                "practice": name,
                "estimate": row["psi"],
                "95% low": row["ci_lower"],
                "95% high": row["ci_upper"],
                "adjusted p": row["p_value_adjusted"],
                "its true effect": truth[f"{key}[{name}]"],
            }
        )
print(pd.DataFrame(rows).set_index(["fit", "practice"]).round(4))
print()
print("navigation's fit now adjusts for:", second.loc["transition_navigation", "adjustment_set"])

Navigation's answer rises from 0.030 to 0.059 (0.032 to 0.086). In this sample the new interval
contains the true 0.0506 and excludes 0.0165, the value with bookings held. The booking's estimate
barely moves (0.0892 against 0.0893), because adjusting for the characteristics and the offer,
without the other practices, is enough for it.

Report each practice from the run whose adjustment matches your question: navigation from the
second run, while the booking's estimate agrees across the two. Adjust each candidate for the others only when none
of them causes another. Holding a later practice fixed leaves out the part of an effect that runs
through it, and can add bias when something unrecorded drives that practice and the outcome. In
this draw the first interval also contains 0.0506, but over 400 simulated repeats of these data
(the multiplicity lesson below describes them) it missed 0.0506 on 254.

## Check the fit

The overlap check reads, for each practice, the chances its fitted treatment model predicts. A
chance near 0 or 1 gives a patient a huge weight. The *effective sample size* says how many
patients the weighted group is worth. The last row is navigation's fit from the second run, the
one the conclusion uses. The combined check (`assess()`) runs the overlap, model and sensitivity checks
on every practice's fit and names any that need attention.

In [ ]:
reports = dict(ranking.diagnostics.support())
# For your own data, name the practice you report from the second run.
reports["navigation, booking follows"] = offer_first.diagnostics.support()["transition_navigation"]
rows = {}
for name, report in reports.items():
    sizes = report.effective_sample_size
    rows[name] = {
        "smallest chance": report.propensity_quantiles["overall"][0.0],
        "largest chance": report.propensity_quantiles["overall"][1.0],
        "share below 0.05": report.tail_mass[0.05]["below"],
        "effective share, got it": sizes["treated"]["ratio"],
        "effective share, did not": sizes["control"]["ratio"],
    }
print(pd.DataFrame(rows).T.round(4).to_string())
print()
plain = {"omitted_confounding": "unmeasured confounding (the sensitivity bound)"}
attention = [plain.get(item.name, item.name) for item in ranking.assess().attention]
print("checks that need attention:", ", ".join(attention) or "none")

The home-health referral has the thinnest overlap: its smallest chance is 0.0070, a share of 0.0448
of patients had less than a 1-in-20 chance of a referral, and the referred patients are worth 54%
of their number after weighting. That is thin but usable: no chance reaches 0 or 1, so every
patient has comparable patients on the other side, and the cost shows in the referral's standard
error, the largest of the five. Chances at 0 or 1 would mean narrowing the question to patients who
could get either option. Navigation's second fit has
smallest chance 0.0361. These chances cannot see anything unrecorded; with your own data, ask staff whether some patients were almost never referred. The one
check that needs attention, unmeasured confounding, is read in the next section; the combined check
skips its simulated-confounder test, which these fits do not support.

## What if a confounder is missing?

How strong would an unrecorded confounder have to be to erase each effect? The robustness value
answers with one number from 0 to 1: the strength that, equal on the outcome side (the share of
the outcome's remaining variation it explains) and on the weight side (how much it would sharpen the
prediction of who got the practice, which enlarges the weights), would pull the estimate to zero. Both strengths run from 0 to 1, like an R
squared. The column at the 95% limit is the strength
at which the interval's low end would reach zero. The benchmark measures how strong a recorded
characteristic is on each side, and the bound gives the range the estimate could move to under a
confounder that strong.

In [ ]:
# For your own data, list each practice you report, with the run you report it from.
fits = {
    "followup_booked": (ranking, "followup_booked"),
    "medication_review": (ranking, "medication_review"),
    "navigation, booking follows": (offer_first, "transition_navigation"),
}
rows = {}
for label, (run, name) in fits.items():
    fitted = run.fits[name]
    robustness = fitted.sensitivity.robustness_value()
    # Measure each recorded characteristic's strength, and keep the one whose bound reaches
    # lowest: a confounder as strong as the strongest recorded one.
    worst = None
    for characteristic in covariates:
        strength = fitted.sensitivity.benchmark(covariates=[characteristic])
        bound = fitted.sensitivity.omitted_confounding(cf_y=strength.cf_y, cf_d=strength.cf_d)
        if worst is None or bound.lower < worst[2].lower:
            worst = (characteristic, strength, bound)
    characteristic, strength, bound = worst
    rows[label] = {
        "robustness value": robustness["rv"],
        "at the 95% limit": robustness["rva"],
        "strongest characteristic": characteristic,
        "its outcome side": strength.cf_y,
        "its weight side": strength.cf_d,
        "bound, low end": bound.lower,
        "bound, high end": bound.upper,
    }
print(pd.DataFrame(rows).T.to_string(float_format=lambda value: f"{value:.3f}"))

The booking's and the review's effects survive a confounder as strong as the strongest recorded
characteristic; navigation's does not. A confounder weaker than 0.094 on both sides at once cannot
erase the booking's effect, one weaker
than 0.050 cannot erase the review's, and one weaker than 0.063 cannot erase navigation's total
effect. A confounder can exceed that number on one side and still fail on the other: for the
booking, `discharge_risk` is 0.099 on the outcome side but 0.028 on the weight side, and the
booking's estimate would stay between 0.042 and 0.137. The review's would stay between 0.012 and
0.076. Navigation's could fall anywhere from -0.088 to 0.206, because teams offered navigation
largely by `discharge_risk`, which is strong on both sides (outcome side 0.108, weight side 0.195).
Only people who know how teams chose could judge whether an unrecorded factor is that strong.

## What we can and can't conclude

For the health plan: booking a follow-up visit before discharge would raise the share who stay
home 30 days by about 9 percentage points (95% interval 0.062 to 0.117). A medication review would
raise it by about 4 points (0.017 to 0.071). Offering navigation, with the visits navigators book,
would raise it by about 6 points (0.032 to 0.086). Teach-back and home-health referrals show no
effect detectable with 4,000 patients.

Each answer is its own causal question with its own conditions, the main one being that nothing
unrecorded drove both that practice and staying home. The booking's and the review's estimates
stay above zero under a confounder as strong as any recorded characteristic; navigation's
would not under one as strong as discharge risk. The effects do not add up to one total. The page
shows one simulated draw with a known answer.

## Go deeper

### How many false leads the adjustment lets through

Testing five practices gives five chances for a practice with no effect to reach p below 0.05 by
luck. The Benjamini-Hochberg adjustment raises the p-values so that, among the practices called
significant, the expected share with no effect, averaged over repeated studies, stays at or below
the chosen level, under the condition given in the ranking above.

In [ ]:
columns = ["p_value", "p_value_adjusted"]
print(table[columns].round(4))

Navigation's raw p-value of 0.0477 is below 0.05, and its adjusted p-value of 0.0794 is above it.
A row's adjusted value depends on the other rows' p-values, so it changes when practices join or
leave the list.

A simulation script kept with this page's review records
(`reviews/notebook-review/probes/ranking-exposures/sweep.py`) repeats both runs on 400 simulated
datasets drawn the same way. Teach-back or the home-health referral reached a raw p-value below
0.05 on 33 of them, and an adjusted one on 20. The mean share of false leads among the practices
called significant was 0.0158 (standard error 0.0035), near 0.02, the bound the adjustment gives
here when its condition holds (0.05 times two no-effect practices out of five). The bound concerns
the average over studies, so one table can hold more false leads, and no simulation study yet
checks it for related fits like these
([X125](../roadmap.md#x125-registered-study-of-variable-importance)).

### Try to break the result

These checks change the data in a known way and refit each practice. Shuffling the practice among
patients (`placebo`) should leave an effect near zero. Adding a column of pure noise
(`random_common_cause`) should barely move the estimate. Refitting on random 70% subsamples of
these patients (`subset`) should give estimates that spread about as much as the `expectation`
column predicts.

In [ ]:
refutation = ranking.diagnostics.refute(tests=("placebo", "random_common_cause", "subset"))
columns = ["candidate", "test", "original", "refuted_mean", "refuted_sd", "expectation", "verdict"]
print(refutation.to_frame()[columns].round(4).to_string(index=False))

With each practice shuffled, the refits average close to zero, such as 0.0061 for the booking
against its estimate of 0.0892. The noise column moves no estimate by more than 0.0002, and the 70%
refits spread about as the `expectation` column predicts, so here no fit reacts oddly to these
known changes. Every `verdict` reads `descriptive`: the simulation studies that set pass and fail
lines use other data and models, so no line applies here. None of these checks can detect a missing
confounder.

### How we know the method works

Each practice's fit is a cross-fitted TMLE of an average treatment effect on a yes-or-no outcome,
and each row prints without an `experimental` mark, which `cleverly` adds where no simulation study
covers a fit.

| section | simulation study | its data and models |
| --- | --- | --- |
| Rank the practices | [stacked point-treatment CV-TMLE](../technical-reference/method-evidence/stacked-point-treatment-cv-tmle.md) | other simulated data, ten folds, unpenalized logistic learners |
| the interval of a difference of two proportions | [asymmetric Wald intervals](../technical-reference/method-evidence/asymmetric-wald-intervals.md) | other simulated data |
| What if a confounder is missing? | [omitted-variable bound standard error](../technical-reference/method-evidence/omitted-variable-bound-standard-error.md), for the limit at the 95% interval | linear simulated data with correct models fitted in sample |
| Try to break the result | [point-treatment refutation rules](../technical-reference/method-evidence/refuter-calibration-point.md) | other simulated data, with unpenalized logistic models, so this page's penalized ones (scikit-learn's default) get no verdict |
| How many false leads the adjustment lets through | none: no study checks the adjusted p-values over related fits | |

The 400-dataset script cited above only checks this page's numbers; it does not validate a
method. The [method
reference](../technical-reference/validation-methods.md#variable-importance) says what the
procedure computes and what it refuses.

## Where to go next

[Rank candidate exposures](../user-guide/variable-importance.md) covers candidates with more than
two levels, ratio effects, and runs that release each fit to save memory. The
[point-treatment example](point-treatment-tmle.ipynb) runs every check on one treatment. [Gain over
current practice](population-impact.ipynb) estimates an effect with a later step set to one level
for everyone, a controlled direct effect; navigation's first row here averages such effects over
the bookings patients actually had. The [examples index](index.md) lists every example.